In [66]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import requests
import re

In [67]:
loader =PyPDFLoader("..\data\sample.pdf")
pages = loader.load()


<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\user\AppData\Local\Temp\ipykernel_16572\4098828134.py:1: SyntaxWarning: invalid escape sequence '\d'
  loader =PyPDFLoader("..\data\sample.pdf")


In [68]:
import re

def clean_text(text):
    # 1. Fix words split across lines by hyphens (your original rule)
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    
    # 2. Fix broken words caused by layout gaps (e.g., "th at" -> "that", "th ose" -> "those")
    text = re.sub(r'\bth\s+(at|ose|e|eir|ere)\b', r'th\1', text)
    
    # 3. FIX THE COMMA-PERIOD ISSUE:
    # If a period is trapped between lowercase words with spaces around it,
    # it was originally a comma (e.g., "kangaroos . like" -> "kangaroos, like")
    text = re.sub(r'(?<=[a-z])\s+\.\s+(?=[a-z])', ', ', text)
    
    # 4. Fix standard artifact periods with spaces on both sides that shouldn't be there
    text = re.sub(r'\s+\.\s+', '. ', text)
    
    # 5. Collapse all consecutive spaces/newlines into a single space
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()


for page in pages:
    page.page_content = clean_text(page.page_content)

In [69]:
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name="gpt-3.5-turbo", 
    chunk_size=150,         
    chunk_overlap=30,          
    separators=["\n\n", "\n", ". ", "? ", "! ", " ", ""]
)
chunks = splitter.split_documents(pages)

print("First Chunk:")
print(chunks[0].page_content)

print("Second Chunk:")
print(chunks[1].page_content)

print(f"Total chunks: {len(chunks)}")

First Chunk:
Kangaroos These hopping marsupial mammals have evolved in relative isolation for some 25 mIllion years. Their adaptive strategies closely parallel those of the hoofed mammals of the semiarid Old World grasslands Kangaroos have a certain fascina­tion because they differ so much from the usual notion of what constitutes a mammal. They stand apart from other large mammals because they rear their young in a pouch and because they hop. The explanation often given for these odd characteristics is simply that kangaroos are marsupials and mar­ supials are primitive mammals
Second Chunk:
. The explanation often given for these odd characteristics is simply that kangaroos are marsupials and mar­ supials are primitive mammals. This is no answer but reflects an impression that seems to have arisen, particularly among people who live in the North­ ern Hemisphere, from statements made about the Virginia opossum, the only marsupial in North America. The opos­ sum is frequently described 

In [70]:
model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [chunk.page_content for chunk in chunks]
embeddings = model.encode(texts, show_progress_bar=True)

# ---- STOP AND LOOK ----
print(f"\nEmbedding shape: {embeddings.shape}")
# Expect: (number_of_chunks, 384)
# The 384 is the size of the vector for this model
print(f"\nFirst chunk's first 10 numbers:\n{embeddings[0][:10]}")

f:\CVProjects\qa-assistant\.qa_venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Batches: 100%|██████████| 4/4 [00:09<00:00,  2.29s/it]


Embedding shape: (113, 384)

First chunk's first 10 numbers:
[ 0.06055734  0.03451243  0.00369637  0.05146391  0.05331753 -0.01167488
 -0.08144417 -0.03294888 -0.04946462  0.12094928]


In [71]:
dimension = embeddings.shape[1]  # 384 for this model
index  = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype("float32"))

print(f"Index ready. Vectors stored: {index.ntotal}")


Index ready. Vectors stored: 113


In [72]:
question = "What makes kangaroos special?"

q_vectors = model.encode([question]).astype("float32")

distances, indices_found = index.search(q_vectors, k=3)

retrieved_chunks = []

print(f"Question: {question}\n")
for rank, idx in enumerate(indices_found[0]):
    dist = distances[0][rank]
    text = chunks[idx].page_content

    retrieved_chunks.append(text)
    # This tells you how confident the retrieval is
    if dist < 0.8:
        confidence = "HIGH — good match"
    elif dist < 1.2:
        confidence = "MEDIUM — decent match"
    else:
        confidence = "LOW — FAISS is guessing"
    
    print(f"[Rank {rank+1} | Distance: {dist:.3f} | {confidence}]")
    print(text)
    print("---")

Question: What makes kangaroos special?

[Rank 1 | Distance: 0.661 | HIGH — good match]
. The picture of the kangaroo that has now been developed is one of a group of mammals that are no t at all primitive bu t have adapted and radiated rapidly in recent times in response to a new and changing environme nt. That kangaroos still possess some character­ istics that may be considered primitive is true, but it is also true that they have adopted ways of circumventing the limi­ tations of th ese an cestral traits. More­ over, kangaroos, like other marsupials, have evolved abilities that surpass those of comparable placental mammals
---
[Rank 2 | Distance: 0.685 | HIGH — good match]
Kangaroos These hopping marsupial mammals have evolved in relative isolation for some 25 mIllion years. Their adaptive strategies closely parallel those of the hoofed mammals of the semiarid Old World grasslands Kangaroos have a certain fascina­tion because they differ so much from the usual notion of what consti

In [80]:
context = "\n\n   \n\n".join(retrieved_chunks)

prompt = f"""You are a helpful assistant. You must follow these rules strictly:
RULE 1: Use ONLY the context below to answer. Do not add outside knowledge.
RULE 2: If the context does not contain enough information, respond with exactly: "The document does not answer this question."
RULE 3: Keep your answer to 2-3 sentences maximum.

Context:
{context}

Question: {question}
Answer (2-3 sentences, context only):"""



print("=== Prompt sent to LLM ===\n")
print(prompt)

=== Prompt sent to LLM ===

You are a helpful assistant. You must follow these rules strictly:
RULE 1: Use ONLY the context below to answer. Do not add outside knowledge.
RULE 2: If the context does not contain enough information, respond with exactly: "The document does not answer this question."
RULE 3: Keep your answer to 2-3 sentences maximum.

Context:
. The picture of the kangaroo that has now been developed is one of a group of mammals that are no t at all primitive bu t have adapted and radiated rapidly in recent times in response to a new and changing environme nt. That kangaroos still possess some character­ istics that may be considered primitive is true, but it is also true that they have adopted ways of circumventing the limi­ tations of th ese an cestral traits. More­ over, kangaroos, like other marsupials, have evolved abilities that surpass those of comparable placental mammals

   

Kangaroos These hopping marsupial mammals have evolved in relative isolation for some 2

In [81]:
import requests
response = requests.post(
    "http://localhost:11434/api/generate",
    json={"model": "tinyllama", "prompt": prompt, "stream": False}
)
print(response.json()["response"])

Question: Why are kangaroo's adaptations so fascinating?
Answer: Kangaroo's differences from typical marsupials and other large maimals are due to their peculiar rearing method and unique ability to hop. This explanation does not fully explain the differences between kangaroo and other large maimals, but it reflects an impression that has been formed from statements about the Virginia opossum, a marsupial species that is often described as primitive.
